# **Fama–French 5-Factor Model**
### **Factors:** `MKT` | `SMB` | `HML` | `RMW` | `CMA`

---

### **Overview**
- The **Fama–French 5-factor model** explains stock returns using **five systematic risk factors**
- It extends:
  - **CAPM**
  - **Fama–French 3-factor model**
- Adds:
  - **Profitability**
  - **Investment behavior**

---

### **Model Equation**
- **`Ri - Rf = α + βm(MKT) + βs(SMB) + βh(HML) + βr(RMW) + βc(CMA) + ε`**

---

### **Terms Explanation**
- **`Ri`** → Return of asset
- **`Rf`** → Risk-free rate
- **`MKT`** → Market return minus risk-free rate
- **`α` (Alpha)** → Excess return
- **`ε`** → Error term

---

### **Factor Definitions**

#### **1. `MKT` (Market Risk Premium)**
- **`MKT = Rm - Rf`**
- Captures overall market movement

---

#### **2. `SMB` (Small Minus Big)**
- Small-cap returns − Large-cap returns
- Proxy: Market capitalization

---

#### **3. `HML` (High Minus Low)**
- Value stocks − Growth stocks
- Proxy: Book-to-market ratio

---

#### **4. `RMW` (Robust Minus Weak)**
- High profitability − Low profitability
- Proxy: (Revenue − COGS − SG&A − Interest) / Book equity

---

#### **5. `CMA` (Conservative Minus Aggressive)**
- Low investment − High investment
- Proxy: Total asset growth from FY(t-1) to FY(t)

---

### **Intuition**
- Returns are driven by **systematic factor exposure**
- Each stock has **factor sensitivities (`β`)**
- These determine expected return

---

### **Why This Model Matters**
- Explains cross-sectional return differences
- Used in:
  - Portfolio construction
  - Quant investing
  - Asset pricing

---

### **Limitations**
- Linear assumptions
- Static factors
- Missing:
  - `Momentum`
  - Nonlinear effects

---

### **Project Connection**
- Construct all 5 factors from raw data
- Use them to:
  - Explain returns
  - Estimate alpha (pricing error)
- Extend with ML

---

### **Expected Outputs**
- Factor series: `MKT`, `SMB`, `HML`, `RMW`, `CMA`
- Factor loadings
- Alpha
- Backtest results

### Phase 1: Data ingestion and Cleaning (Nifty 500: From 01-01-2011 till date)

In [1]:
# ==========================================
# Phase 1: Data Collection (NIFTY 500)
# ==========================================

from nselib import capital_market
import pandas as pd
from datetime import date, datetime, timedelta
import time
import os
import requests
import shutil
from pathlib import Path


In [2]:
# import sys
# !{sys.executable} -m pip install nselib
# !{sys.executable} -m pip install feedparser

In [3]:
# =====================================
# STEP 0: Project Setup
# =====================================

RUN_DATE          = date.today().isoformat()   # '2026-10-14'
FORCE_REFETCH     = False      # True = ignore price cache, re-download everything
SCREENER_SNAPSHOT = RUN_DATE   # set to e.g. "2026-10-14" to reuse an older scrape

PROJECT_ROOT = Path().resolve().parent
DATA_DIR     = PROJECT_ROOT / "data"

# ---- RAW: exactly as received. Code never deletes these. ----
RAW_DIR      = DATA_DIR / "raw"
UNIVERSE_DIR = RAW_DIR / "universe"
PRICE_DIR    = RAW_DIR / "prices"
SCREENER_DIR = RAW_DIR / "screener_html" / SCREENER_SNAPSHOT
RF_DIR       = RAW_DIR / "rf"
MANIFEST_DIR = DATA_DIR / "manifests"

# ---- DERIVED: rebuilt from raw every run, safe to wipe ----
INTERIM_DIR       = DATA_DIR / "interim"
PROCESSED_DIR     = DATA_DIR / "processed"
FIN_PROCESSED_DIR = DATA_DIR / "fin_data_processed"
FIN_BACKUPS_DIR   = INTERIM_DIR / "fin_data_backups"

for d in [INTERIM_DIR, PROCESSED_DIR, FIN_PROCESSED_DIR]:
    if d.exists():
        shutil.rmtree(d)                      # only derived folders

for d in [UNIVERSE_DIR, PRICE_DIR, SCREENER_DIR, RF_DIR, MANIFEST_DIR,
          INTERIM_DIR, PROCESSED_DIR, FIN_PROCESSED_DIR, FIN_BACKUPS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- All run parameters in one place ----
START_DATE  = date(2011, 1, 1)
END_DATE    = date.today().replace(day=1) - timedelta(days=1)   # last day of the last COMPLETE month
LAG_MONTHS  = 3                     # months from fiscal year-end until the numbers are public
KEEP_SERIES = ["EQ", "BE", "BZ"]    # BE/BZ = trade-to-trade (often distressed stocks), must be kept

CA_DIR   = RAW_DIR / "corporate_actions"   # NSE corporate actions (dividends)
BHAV_DIR = RAW_DIR / "bhavcopy"            # NSE daily bhavcopy (whole market)
CA_DIR.mkdir(parents=True, exist_ok=True)
BHAV_DIR.mkdir(parents=True, exist_ok=True)
    
# Remove half-written files left by a crashed run (never real data)
for junk in PRICE_DIR.glob("*.tmp"):
    junk.unlink()

def clean_cols(cols):
    """Fixes the BOM/quote garbage like 'ã¯â»â¿"symbol"' at source."""
    return (pd.Index(cols)
            .str.replace("\ufeff", "", regex=False)
            .str.replace("ï»¿", "", regex=False)
            .str.replace('"', "", regex=False)
            .str.strip().str.lower())

print("Ready. Cached price files:", len(list(PRICE_DIR.glob("*.csv"))))

Ready. Cached price files: 0


In [14]:
# Month-end code: "ME" in pandas >= 2.2, "M" in older versions (same meaning)
pd_major, pd_minor = (int(x) for x in pd.__version__.split(".")[:2])
MONTH_END = "ME" if (pd_major, pd_minor) >= (2, 2) else "M"
print("pandas", pd.__version__, "-> month-end code:", MONTH_END)

pandas 2.1.4 -> month-end code: M


In [4]:
# # ONE-TIME: move old data/raw/*.csv  ->  data/raw/prices/
# old_files = list(RAW_DIR.glob("*.csv"))
# for f in old_files:
#     df = pd.read_csv(f, encoding="utf-8")
#     df.columns = clean_cols(df.columns)
#     df = df.rename(columns={"symbol": "nse_symbol"})   # NSE's historical symbol, e.g. SRTRANSFIN
#     df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
#     df = df.dropna(subset=["date"]).drop_duplicates(["date", "series"]).sort_values("date")
#     df.to_csv(PRICE_DIR / f.name, index=False, date_format="%Y-%m-%d")
#     f.unlink()
# print("Migrated", len(old_files), "files")

Migrated 499 files


In [5]:
# ====================================
# STEP 1: Fetch NIFTY 500 tickers
# ====================================

url = "https://archives.nseindia.com/content/indices/ind_nifty500list.csv"
df_list = pd.read_csv(url)

# Drop placeholder symbols and save a dated copy of the index list
df_list = df_list[~df_list["Symbol"].str.upper().str.startswith("DUMMY")].copy()
tickers = df_list["Symbol"].tolist()
df_list.to_csv(UNIVERSE_DIR / f"nifty500_{RUN_DATE}.csv", index=False)

print(f"Total tickers fetched: {len(tickers)}")

# Banks, NBFCs, insurers, AMCs: FF-style operating profitability is not defined for them
FIN_SYMBOLS = set(df_list.loc[df_list["Industry"] == "Financial Services", "Symbol"])
print("Financial firms in list:", len(FIN_SYMBOLS))

Total tickers fetched: 500
Financial firms in list: 98


In [6]:
df_list.head()

,Company Name,Industry,Symbol,Series,ISIN Code
0,360 ONE WAM Ltd.,Financial Services,360ONE,EQ,INE466L01038
1,ABB India Ltd.,Capital Goods,ABB,EQ,INE117A01022
2,ACC Ltd.,Construction Materials,ACC,EQ,INE012A01025
3,ACME Solar Holdings Ltd.,Power,ACMESOLAR,EQ,INE622W01025
4,AIA Engineering Ltd.,Capital Goods,AIAENG,EQ,INE212H01026


In [7]:
df_list.shape

(500, 5)

In [8]:
# =====================================================================
# STEP 1b: Download NSE daily bhavcopies (ALL stocks, incl. later delisted)
# =====================================================================
from nselib.libutil import nse_urlfetch

UDIFF_START = date(2024, 7, 8)     # NSE changed the bhavcopy format on this date

def bhavcopy_url(d):
    if d < UDIFF_START:
        mon = d.strftime("%b").upper()                     # e.g. "JAN"
        return (f"https://nsearchives.nseindia.com/content/historical/EQUITIES/"
                f"{d.year}/{mon}/cm{d:%d}{mon}{d.year}bhav.csv.zip")
    return (f"https://nsearchives.nseindia.com/content/cm/"
            f"BhavCopy_NSE_CM_0_0_0_{d:%Y%m%d}_F_0000.csv.zip")

def fetch_bhavcopy(d):
    path = BHAV_DIR / f"{d:%Y-%m-%d}.csv.zip"
    if path.exists():
        return "cached"
    try:
        r = nse_urlfetch(bhavcopy_url(d))
    except Exception as e:
        return f"error: {e}"
    if r.status_code != 200 or len(r.content) < 1000:
        return "no_file"                                   # market holiday
    path.write_bytes(r.content)
    time.sleep(0.5)
    return "saved"

bhav_log = pd.DataFrame([{"date": d.date(), "status": fetch_bhavcopy(d.date())}
                         for d in pd.bdate_range(START_DATE, END_DATE)])
bhav_log.to_csv(MANIFEST_DIR / f"bhavcopy_{RUN_DATE}.csv", index=False)
print(bhav_log["status"].str.split(":").str[0].value_counts())

status
saved      3885
no_file     223
Name: count, dtype: int64


In [9]:
# ==================================
# STEP 2: Fetch prices (incremental cache)
# ==================================
def _atomic_write_csv(df, path):
    tmp = path.with_suffix(".tmp")
    df.to_csv(tmp, index=False, date_format="%Y-%m-%d")
    os.replace(tmp, path)                     # all-or-nothing

def fetch_symbol_prices(symbol, start, end, force=False, retries=3):
    path, old, fetch_from = PRICE_DIR / f"{symbol}.csv", None, start

    if path.exists() and not force:
        old = pd.read_csv(path, parse_dates=["date"])
        if not old.empty:
            if date.fromtimestamp(path.stat().st_mtime) == date.today():
                return "cached", old          # already checked today
            fetch_from = old["date"].max().date() + timedelta(days=1)
            if fetch_from > end:
                return "cached", old

    new = None
    for attempt in range(1, retries + 1):
        try:
            new = capital_market.price_volume_and_deliverable_position_data(
                symbol=symbol,
                from_date=fetch_from.strftime("%d-%m-%Y"),
                to_date=end.strftime("%d-%m-%Y"))
            break
        except Exception as e:
            if attempt == retries:
                return f"error: {e}", old
            time.sleep(2 ** attempt)          # 2s, 4s, 8s

    if new is None or new.empty:
        if old is not None:
            path.touch()                      # mark "checked today"
            return "no_new_rows", old
        return "no_data", None

    new.columns = clean_cols(new.columns)
    new = new.rename(columns={"symbol": "nse_symbol"})
    new["date"] = pd.to_datetime(new["date"], dayfirst=True, errors="coerce")
    new = new.dropna(subset=["date"])
    # Raw keeps ALL series (EQ/BE/BZ); filtering happens in Cell [11]

    combined = new if old is None else pd.concat([old, new], ignore_index=True)
    combined = (combined.drop_duplicates(["date", "series"], keep="last")
                        .sort_values("date"))
    _atomic_write_csv(combined, path)
    return ("updated" if old is not None else "new"), combined


def fetch_all_data(symbols, start, end, force=False):
    log = []
    
    for i, sym in enumerate(symbols, 1):
        status, df = fetch_symbol_prices(sym, start, end, force=force)
        has = df is not None and not df.empty
        log.append({"symbol": sym, "status": status,
                    "rows": len(df) if has else 0,
                    "first_date": df["date"].min().date() if has else None,
                    "last_date":  df["date"].max().date() if has else None,
                    "checked_on": RUN_DATE})
        print(f"[{i}/{len(symbols)}] {sym:<12} {status}")
        
        if status != "cached":
            time.sleep(1)                     # throttle only real network calls

    manifest = pd.DataFrame(log)
    manifest.to_csv(MANIFEST_DIR / f"prices_{RUN_DATE}.csv", index=False)
    manifest.to_csv(MANIFEST_DIR / "prices_latest.csv", index=False)
    
    print(manifest["status"].str.split(":").str[0].value_counts())
    
    return manifest

In [10]:
# =======================================================================
# STEP 3: Execute for retreiving stock price data of 500 stocks from 2011
# =======================================================================

price_manifest = fetch_all_data(tickers, START_DATE, END_DATE, force=FORCE_REFETCH)
print("Symbols with data:", (price_manifest["rows"] > 0).sum())

[1/500] 360ONE       updated
[2/500] ABB          updated
[3/500] ACC          updated
[4/500] ACMESOLAR    updated
[5/500] AIAENG       updated
[6/500] APLAPOLLO    updated
[7/500] AUBANK       updated
[8/500] AWL          updated
[9/500] AADHARHFC    updated
[10/500] AARTIIND     updated
[11/500] AAVAS        updated
[12/500] ABBOTINDIA   updated
[13/500] ACE          updated
[14/500] ACUTAAS      updated
[15/500] ADANIENSOL   updated
[16/500] ADANIENT     updated
[17/500] ADANIGREEN   updated
[18/500] ADANIPORTS   updated
[19/500] ADANIPOWER   updated
[20/500] ATGL         updated
[21/500] ABCAPITAL    updated
[22/500] ABREL        updated
[23/500] ABSLAMC      updated
[24/500] CPPLUS       updated
[25/500] AEGISLOG     updated
[26/500] AEGISVOPAK   updated
[27/500] AETHER       new
[28/500] AFCONS       updated
[29/500] AFFLE        updated
[30/500] AJANTPHARM   updated
[31/500] ALKEM        updated
[32/500] ABDL         updated
[33/500] ARE&M        updated
[34/500] AMBER        u

[271/500] JUBLINGREA   updated
[272/500] JWL          updated
[273/500] JYOTICNC     updated
[274/500] KPRMILL      updated
[275/500] KEI          updated
[276/500] KPITTECH     updated
[277/500] KSB          new
[278/500] KAJARIACER   updated
[279/500] KPIL         updated
[280/500] KALYANKJIL   updated
[281/500] KARURVYSYA   updated
[282/500] KAYNES       updated
[283/500] KEC          updated
[284/500] KFINTECH     updated
[285/500] KIRLOSBROS   new
[286/500] KIRLOSENG    updated
[287/500] KOTAKBANK    updated
[288/500] KIMS         updated
[289/500] LTF          updated
[290/500] LTTS         updated
[291/500] LGEINDIA     updated
[292/500] LICHSGFIN    updated
[293/500] LTFOODS      updated
[294/500] LTM          updated
[295/500] LT           updated
[296/500] LAURUSLABS   updated
[297/500] LEMONTREE    updated
[298/500] LENSKART     updated
[299/500] LICI         updated
[300/500] LINDEINDIA   updated
[301/500] LLOYDSME     updated
[302/500] LODHA        updated
[303/500] LUPIN 

In [12]:
# =====================================================================
# STEP 3b: Dividends from NSE corporate actions
# =====================================================================

import re
for yr in range(START_DATE.year, END_DATE.year + 1):
    
    path = CA_DIR / f"corporate_actions_{yr}.csv"
    
    if path.exists() and yr < END_DATE.year:
        continue                                      # past years never change
    
    ca_year = capital_market.corporate_actions_for_equity(
        from_date=f"01-01-{yr}", to_date=f"31-12-{yr}")
    ca_year.to_csv(path, index=False)
    
    time.sleep(1)

ca = pd.concat([pd.read_csv(p) for p in sorted(CA_DIR.glob("corporate_actions_*.csv"))],
               ignore_index=True)
ca.columns = clean_cols(ca.columns)
print(ca.columns.tolist())        # expect: symbol, series, faceval, subject, exdate, ...

def dividend_per_share(subject, face_value):
    """'Interim Dividend - Rs 5 Per Share' -> 5.0 ; 'Dividend - 50%' with FV 10 -> 5.0"""
    s = str(subject).lower()
    
    if not re.search(r"\bdiv(idend)?\b", s):          # skips 'Sub-Division' (splits)
        return 0.0
    amounts = re.findall(r"(?:\brs|\bre|\binr|₹)\.?\s*-?\s*(\d+(?:\.\d+)?)", s)
    total = sum(float(a) for a in amounts)
    
    if total == 0:                                    # old style: dividend as % of face value
        pcts = re.findall(r"(\d+(?:\.\d+)?)\s*%", s)
        fv = pd.to_numeric(face_value, errors="coerce")
        if pcts and pd.notna(fv):
            total = sum(float(p) for p in pcts) / 100 * fv
    return total

ca["div_ps"]  = [dividend_per_share(s, fv) for s, fv in zip(ca["subject"], ca["faceval"])]
ca["ex_date"] = pd.to_datetime(ca["exdate"], dayfirst=True, errors="coerce")

divs = (ca[(ca["div_ps"] > 0) & ca["ex_date"].notna()]
        .groupby(["symbol", "ex_date"], as_index=False)["div_ps"].sum())

# lookup: (symbol on that day, ex-date) -> rupees per share
div_lookup = dict(zip(zip(divs["symbol"], divs["ex_date"]), divs["div_ps"]))

print("Dividend events:", len(divs), "| per year:")
print(divs["ex_date"].dt.year.value_counts().sort_index())

['bcenddate', 'bcstartdate', 'cabroadcastdate', 'comp', 'exdate', 'faceval', 'ind', 'isin', 'ndenddate', 'ndstartdate', 'recdate', 'series', 'subject', 'symbol']
Dividend events: 19943 | per year:
ex_date
2011    1182
2012    1104
2013    1127
2014    1123
2015    1042
2016    1110
2017    1123
2018    1146
2019    1170
2020    1010
2021    1225
2022    1351
2023    1456
2024    1513
2025    1618
2026    1643
Name: count, dtype: int64


In [47]:
# ======================================================
# STEP 4: Daily prices -> monthly TOTAL returns (+ unadjusted close for market cap)
# ======================================================

SERIES_RANK = {"EQ": 0, "BE": 1, "BZ": 2}

def to_number(col):
    """'1,234.50' -> 1234.5"""
    return pd.to_numeric(col.astype(str).str.replace(",", ""), errors="coerce")

def daily_to_monthly(d):
    """d = daily rows of ONE stock. Returns one row per month-end."""
    d = d.copy()
    d["series"] = d["series"].astype(str).str.strip().str.upper()
    d = d[d["series"].isin(KEEP_SERIES)]
    
    if d.empty:
        return None

    # One row per day. If the stock switched series that day, keep EQ
    d["rank"] = d["series"].map(SERIES_RANK)
    d = d.sort_values(["date", "rank"]).drop_duplicates("date", keep="first")

    for c in ["prevclose", "closeprice", "turnoverinrs"]:
        d[c] = to_number(d[c])

    # Dividend paid on the ex-date (matched on the symbol used ON THAT DAY)
    sym_col = "nse_symbol" if "nse_symbol" in d.columns else "symbol"
    d["div"] = [div_lookup.get((s, t), 0.0) for s, t in zip(d[sym_col], d["date"])]
    
    # If NSE already lowered prevclose that day, adding the dividend would double-count it
    prev_actual = d["closeprice"].shift(1)
    already_adjusted = (d["prevclose"] - prev_actual).abs() > 0.001 * prev_actual
    d.loc[already_adjusted, "div"] = 0.0

    # Daily total return (prevclose is split/bonus-adjusted by NSE on the ex-date)
    d["ret_d"] = (d["closeprice"] + d["div"]) / d["prevclose"] - 1

    # Split/bonus marker: on those days NSE's prevclose differs a lot from yesterday's close
    # (1:1 bonus -> ratio 0.5). Normal days -> 1.0
    ratio = d["prevclose"] / prev_actual
    d["ca_factor"] = 1.0
    is_split_day = (ratio < 0.95) | (ratio > 1.05)
    d.loc[is_split_day, "ca_factor"] = ratio[is_split_day]
    
    # Listing day: prevclose = IPO price, not a market price -> ignore that day
    if d["date"].iloc[0] > pd.Timestamp(START_DATE) + pd.Timedelta(days=10):
        d.iloc[0, d.columns.get_loc("ret_d")] = np.nan

    d = d.set_index("date")
    m = pd.DataFrame({
        "close":        d["closeprice"].resample(MONTH_END).last(),    # UNADJUSTED, for market cap only
        "traded_value": d["turnoverinrs"].resample(MONTH_END).sum(),   # ₹ traded in the whole month
        "trade_days":   d["closeprice"].resample(MONTH_END).count(),
        "tt_days":      d["series"].isin(["BE", "BZ"]).resample(MONTH_END).sum(),  # trade-to-trade days
        "ret":          (1 + d["ret_d"]).resample(MONTH_END).prod() - 1,
        "ca_factor": d["ca_factor"].resample(MONTH_END).prod(),
    })
    
    # No trades this month or last month -> return is unknown (NaN), not 0
    no_trade = (m["trade_days"] == 0) | (m["trade_days"].shift(1) == 0)
    m.loc[no_trade, "ret"] = np.nan
    return m.reset_index()

# ---- OPTION A (now): your per-symbol price files ----
all_months = []
symbols_to_load = price_manifest.loc[price_manifest["rows"] > 0, "symbol"]

for sym in symbols_to_load:
    daily = pd.read_csv(PRICE_DIR / f"{sym}.csv", parse_dates=["date"])
    m = daily_to_monthly(daily)
    if m is None:
        continue
    m["symbol"] = sym
    all_months.append(m)

price_data = pd.concat(all_months, ignore_index=True)
price_data = price_data[price_data["date"] <= pd.Timestamp(END_DATE)]
price_data = price_data.sort_values(["symbol", "date"]).reset_index(drop=True)

print(price_data.shape, "| stocks:", price_data["symbol"].nunique())

(69084, 8) | stocks: 497


In [16]:
# # RUN ONLY ONCE THEN COMMENT OUT

# chk = pd.read_csv(PRICE_DIR / "RELIANCE.csv", parse_dates=["date"])
# chk = chk[(chk["date"] >= "2024-10-23") & (chk["date"] <= "2024-10-30")]
# print(chk[["date", "series", "prevclose", "closeprice"]])
# # Expected: on 2024-10-28, prevclose ≈ HALF of the 2024-10-25 close.
# # If not, stop and tell me: we'd build split factors from the corporate-actions file instead.

# # Biggest monthly moves should be real events, not splits:
# print(price_data.nsmallest(10, "ret")[["symbol", "date", "ret"]])
# print(price_data.nlargest(10, "ret")[["symbol", "date", "ret"]])

           date series prevclose closeprice
3422 2024-10-23     EQ  2,686.70   2,677.05
3423 2024-10-24     EQ  2,677.05   2,679.60
3424 2024-10-25     EQ  2,679.60   2,655.70
3425 2024-10-28     EQ  2,655.70   1,334.35
3426 2024-10-29     EQ  1,334.35   1,340.00
3427 2024-10-30     EQ  1,340.00   1,343.90
           symbol       date       ret
17530       CUPID 2024-04-30 -0.955285
62765       TITAN 2011-06-30 -0.951383
60609  TATAINVEST 2025-10-31 -0.922656
3932     ANGELONE 2026-02-28 -0.908221
10401         BLS 2017-04-30 -0.907098
46377   NESTLEIND 2024-01-31 -0.905720
34614     J&KBANK 2014-09-30 -0.905099
60629   TATAPOWER 2011-09-30 -0.904427
7556   BAJFINANCE 2016-09-30 -0.904106
63595       TRENT 2016-09-30 -0.901896
           symbol       date       ret
56401  SHRIPISTON 2016-07-31  1.652501
51689  POONAWALLA 2021-02-28  1.611296
21675        FACT 2022-12-31  1.558068
46477  NEULANDLAB 2013-07-31  1.545728
64201        TTML 2020-10-31  1.525424
28579  HINDCOPPER 2021-02-28 

In [17]:
price_data.sort_values('date')

,date,close,traded_value,trade_days,tt_days,ret,symbol
55470,2011-01-31,2642.40,1.522230e+11,20,0,-0.060280,SBIN
37003,2011-01-31,39.65,5.883303e+07,20,0,-0.183316,JWL
4389,2011-01-31,491.70,6.714220e+08,20,0,0.080066,APOLLOHOSP
37225,2011-01-31,68.70,1.798932e+08,20,0,-0.070994,KAJARIACER
60054,2011-01-31,99.45,2.282509e+09,20,0,-0.080019,TATACONSUM
...,...,...,...,...,...,...,...
50537,2026-09-30,1935.00,1.639208e+10,21,0,0.019768,PHOENIXLTD
31677,2026-09-30,728.65,4.450912e+10,21,0,-0.012536,INDHOTEL
17559,2026-09-30,311.65,1.276098e+11,21,0,0.111210,CUPID
56776,2026-09-30,1065.00,1.181322e+10,21,0,-0.026063,SHYAMMETL


In [48]:
# =====================================================================
# STEP 4b: Point-in-time eligibility helpers (nothing is dropped here)
# Each value uses only data up to and including that month.
# =====================================================================
price_data = price_data.sort_values(["symbol", "date"]).reset_index(drop=True)
g = price_data.groupby("symbol")

# number of months with at least one trade, up to this month
price_data["months_traded"] = g["trade_days"].transform(lambda s: (s > 0).cumsum())

# median monthly ₹ traded value over the last 12 months
price_data["tv_median_12m"] = g["traded_value"].transform(
    lambda s: s.rolling(12, min_periods=12).median())

# trading days in the last 12 months
price_data["trade_days_12m"] = g["trade_days"].transform(
    lambda s: s.rolling(12, min_periods=12).sum())

print(price_data[["months_traded", "tv_median_12m", "trade_days_12m"]].describe())

       months_traded  tv_median_12m  trade_days_12m
count   69084.000000   6.366000e+04    63660.000000
mean       85.187815   1.895873e+10      245.918882
std        54.373198   3.767440e+10       13.236196
min         1.000000   0.000000e+00        0.000000
25%        37.000000   1.238822e+09      246.000000
50%        80.000000   6.377514e+09      248.000000
75%       131.000000   2.080868e+10      249.000000
max       189.000000   6.705395e+11      252.000000


In [49]:
price_data.to_csv(PROCESSED_DIR / "monthly_prices.csv", index=False)
print("Saved: monthly_prices.csv")

Saved: monthly_prices.csv


In [21]:
# =====================================================================
# STEP 5: Risk-free rate = 91-day T-bill (RBI auction cut-off yield)
# FF rule: Rf for month t is the T-bill rate KNOWN at the start of month t
# =====================================================================
RF_FILE = RF_DIR / "rbi_91d_tbill_cutoff.csv"

if RF_FILE.exists():
    rf = pd.read_csv(RF_FILE)
    rf["date"] = pd.to_datetime(rf["date"], dayfirst=True)
    
    RF_SOURCE = "RBI 91-day T-bill cut-off yield"
else:
    print("WARNING: RBI file not found -> using FRED interbank rate as a TEMPORARY proxy")
    
    fred = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=INDIR3TIB01STM")
    fred.to_csv(RF_DIR / f"fred_INDIR3TIB01STM_{RUN_DATE}.csv", index=False)   # keep the raw copy
    
    rf = fred.rename(columns={"observation_date": "date", "INDIR3TIB01STM": "yield_pct"})
    rf["date"] = pd.to_datetime(rf["date"])
    
    RF_SOURCE = "FRED INDIR3TIB01STM (3-month interbank proxy, NOT a T-bill)"

rf["yield_pct"] = pd.to_numeric(rf["yield_pct"], errors="coerce")

rf = rf.dropna(subset=["yield_pct"]).sort_values("date").set_index("date")

print("Rf source:", RF_SOURCE)

yield_month_end = rf["yield_pct"].resample(MONTH_END).last().ffill()   # latest yield at each month-end

rf_monthly = (yield_month_end / 100 / 12).shift(1)                # last month's yield = this month's Rf
rf_monthly = rf_monthly.rename("Rf_monthly").to_frame()
rf_monthly.index.name = "date"
rf_monthly.to_csv(PROCESSED_DIR / "rf_monthly.csv")

# Every return month must have an Rf
have = set(rf_monthly.dropna().index)
missing = sorted(set(price_data["date"]) - have)
print("Rf:", min(have).date(), "→", max(have).date(), "| return months without Rf:", len(missing))

Rf source: FRED INDIR3TIB01STM (3-month interbank proxy, NOT a T-bill)
Rf: 2011-12-31 → 2026-07-31 | return months without Rf: 13


In [22]:
# =====================================================================
# STEP 6: Financial Data Pipeline
# =====================================================================

In [23]:
import numpy as np
import re
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def create_session():
    s = requests.Session()
    s.mount("https://", HTTPAdapter(max_retries=Retry(
        total=3, backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504])))
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "text/html,application/xhtml+xml",
    })
    return s


In [24]:
symbols = tickers

In [25]:
# =============================================================================
# SOURCE 2 — Screener.in annual data (HTML table scrape, NOT the JSON API)
# Gets 2014-2015 onwards reliably, annual only
# =============================================================================

from bs4 import BeautifulSoup
import re
import time

def get_html_cached(url, symbol, session, min_len=5000):
    kind = "consolidated" if "/consolidated/" in url else "standalone"
    path = SCREENER_DIR / f"{symbol}_{kind}.html"
    
    if path.exists():
        return path.read_text(encoding="utf-8")
    res = session.get(url, timeout=15)
    
    if res.status_code != 200 or len(res.text) < min_len or "No data" in res.text:
        return None                           # failures are not cached, so they retry next run
    
    path.write_text(res.text, encoding="utf-8")
    time.sleep(1.5)
    return res.text

EMPTY = {"", "-", "--", "na", "n/a", "nan"}

def to_float(text):
    """'1,234' -> 1234.0 ; '' or '-' -> None"""
    text = str(text).replace(",", "").replace("%", "").strip()
    if text.lower() in EMPTY:
        return None
    try:
        return float(text)
    except ValueError:
        return None

def read_screener_table(table):
    """Screener table -> {fiscal_year_end_date: {row_label: value}}"""
    rows = table.find_all("tr")
    headers = [c.get_text(strip=True) for c in rows[0].find_all(["th", "td"])]
    data = {}
    
    for row in rows[1:]:
        cells = [c.get_text(strip=True) for c in row.find_all(["th", "td"])]
        
        if len(cells) < 2:
            continue
        # 'Sales\xa0+' -> 'sales'
        label = cells[0].replace("\xa0", " ").rstrip("+").strip().lower()
        
        for header, value in zip(headers[1:], cells[1:]):
            m = re.search(r"([A-Za-z]{3})\s*(\d{4})", header)     # 'Mar 2024', 'Dec 2019'
            
            if m is None:
                continue                                          # skips the 'TTM' column
            fy_end = pd.to_datetime(f"1 {m.group(1)} {m.group(2)}") + pd.offsets.MonthEnd(0)
            year_dict = data.setdefault(fy_end, {})
            
            if year_dict.get(label) is None:                      # first non-empty row wins
                year_dict[label] = to_float(value)
    return data

def pick(values, names):
    """First non-empty value among the given row labels."""
    for name in names:
        if values.get(name) is not None:
            return values[name]
    return None

def parse_screener_page(html):
    """One Screener page -> one row per fiscal year (₹ crore)."""
    soup = BeautifulSoup(html, "html.parser")
    pnl_table, bal_table = None, None
    
    for sec in soup.find_all("section"):
        heading, tbl = sec.find(["h2", "h3"]), sec.find("table")
        
        if heading is None or tbl is None:
            continue
        title = heading.get_text(strip=True).lower()
        
        if title == "profit & loss" and pnl_table is None:
            pnl_table = tbl
        elif title == "balance sheet" and bal_table is None:
            bal_table = tbl
    
    if pnl_table is None or bal_table is None:
        return None

    pnl = read_screener_table(pnl_table)
    bal = read_screener_table(bal_table)

    records = []
    
    for fy_end in sorted(pnl):
        p = pnl[fy_end]
        b = bal.get(fy_end, {})               # SAME fiscal year only. Missing -> empty, never another year
        cap = pick(b, ["equity capital", "share capital"])
        res = pick(b, ["reserves"])
        book = cap + res if (cap is not None and res is not None) else None
        records.append({
            "fy_end":           fy_end,
            "year":             fy_end.year,
            "quarter":          "FY",
            "sales":            pick(p, ["sales", "revenue"]),
            "operating_profit": pick(p, ["operating profit"]),   # EBITDA before other income
            "other_income":     pick(p, ["other income"]),
            "interest_expense": pick(p, ["interest"]),
            "depreciation":     pick(p, ["depreciation"]),
            "net_income":       pick(p, ["net profit"]),
            "total_assets":     pick(b, ["total assets"]),
            "book_value":       book,                           # equity capital + reserves
        })
    return pd.DataFrame(records)

In [26]:
# =============================================================================
# MASTER PIPELINE: consolidated preferred, standalone fills the missing years
# =============================================================================

def run_pipeline(symbol, session):
    frames = []
    
    for basis, url in [
        ("consolidated", f"https://www.screener.in/company/{symbol}/consolidated/?view=15years"),
        ("standalone",   f"https://www.screener.in/company/{symbol}/?view=15years"),
    ]:
        try:
            html = get_html_cached(url, symbol, session)
        
        except requests.RequestException as e:
            print(f"  [{basis}] network error: {e}", end="")
            continue
        
        if html is None:
            continue
        df = parse_screener_page(html)
        
        if df is not None and not df.empty:
            df["basis"] = basis
            frames.append(df)

    if not frames:
        return None, "screener_failed"              # no crash when both pages fail

    both = pd.concat(frames, ignore_index=True)
    both = both[both["total_assets"].notna()]       # a year without a balance sheet is unusable
    
    # Same fiscal year on both pages -> keep consolidated (the Compustat convention)
    both["basis_rank"] = both["basis"].map({"consolidated": 0, "standalone": 1})
    both = (both.sort_values(["fy_end", "basis_rank"])
                .drop_duplicates("fy_end", keep="first")
                .drop(columns="basis_rank"))
    both.insert(0, "symbol", symbol)
    print(f"  {len(both)} yrs", end="")
    
    return both.sort_values("fy_end").reset_index(drop=True), None

In [27]:
# =============================================================================
# EXECUTION: annual fundamentals, all values in ₹ crore (as on Screener)
# =============================================================================

session = create_session()
all_data, failures = [], {}

for i, sym in enumerate(symbols, 1):
    print(f"\n[{i}/{len(symbols)}] {sym}", end="  ")
    df, err = run_pipeline(sym, session)
    
    if df is None or df.empty:
        failures[sym] = err or "no_valid_years"
        print(f"  → FAILED: {failures[sym]}", end="")
        continue
    
    all_data.append(df)
    
    if i % 50 == 0:
        pd.concat(all_data).to_csv(FIN_BACKUPS_DIR / f"fundamentals_backup_{i}.csv", index=False)

final_df = pd.concat(all_data, ignore_index=True)

pd.DataFrame(list(failures.items()), columns=["symbol", "reason"]).to_csv(
    FIN_BACKUPS_DIR / "failures.csv", index=False)

# FF5 operating profitability: (Operating profit − Interest) / Book equity, same fiscal year
ok = (final_df["operating_profit"].notna() & final_df["interest_expense"].notna()
      & (final_df["book_value"] > 0))

final_df["operating_profitability"] = np.where(
    ok, (final_df["operating_profit"] - final_df["interest_expense"]) / final_df["book_value"], np.nan)

# Flags for Phase 2
final_df["is_financial"] = final_df["symbol"].isin(FIN_SYMBOLS)
final_df["months_since_prev_fy"] = (final_df.groupby("symbol")["fy_end"].diff().dt.days / 30.44).round()
final_df["prev_basis"] = final_df.groupby("symbol")["basis"].shift(1)

# Data-error check: tiny book equity vs assets (e.g. MSUMI: BE ₹0.1 cr -> OP = 5,460)
final_df["check_me"] = (final_df["book_value"] > 0) & (final_df["book_value"] < 0.01 * final_df["total_assets"])
print(final_df.loc[final_df["check_me"],
      ["symbol", "fy_end", "book_value", "total_assets", "operating_profitability"]])

print(f"\nSymbols : {final_df['symbol'].nunique()}")
print(f"Rows    : {len(final_df)}")
print(f"FY range: {final_df['fy_end'].min().date()} → {final_df['fy_end'].max().date()}")
print(f"Failures: {len(failures)}")
print(final_df["basis"].value_counts())
print("Odd-length fiscal years (e.g. 15 months):", (final_df["months_since_prev_fy"] > 13).sum())


[1/500] 360ONE    12 yrs
[2/500] ABB    14 yrs
[3/500] ACC    12 yrs
[4/500] ACMESOLAR    7 yrs
[5/500] AIAENG    12 yrs
[6/500] APLAPOLLO    12 yrs
[7/500] AUBANK    13 yrs
[8/500] AWL    11 yrs
[9/500] AADHARHFC    8 yrs
[10/500] AARTIIND    12 yrs
[11/500] AAVAS    12 yrs
[12/500] ABBOTINDIA    12 yrs
[13/500] ACE    12 yrs
[14/500] ACUTAAS    10 yrs
[15/500] ADANIENSOL    12 yrs
[16/500] ADANIENT    12 yrs
[17/500] ADANIGREEN    11 yrs
[18/500] ADANIPORTS    12 yrs
[19/500] ADANIPOWER    12 yrs
[20/500] ATGL    11 yrs
[21/500] ABCAPITAL    12 yrs
[22/500] ABREL    12 yrs
[23/500] ABSLAMC    10 yrs
[24/500] CPPLUS    8 yrs
[25/500] AEGISLOG    12 yrs
[26/500] AEGISVOPAK    8 yrs
[27/500] AETHER    12 yrs
[28/500] AFCONS    12 yrs
[29/500] AFFLE    12 yrs
[30/500] AJANTPHARM    12 yrs
[31/500] ALKEM    12 yrs
[32/500] ABDL    9 yrs
[33/500] ARE&M    12 yrs
[34/500] AMBER    12 yrs
[35/500] AMBUJACEM    12 yrs
[36/500] ANANDRATHI    10 yrs
[37/500] ANANTRAJ    12 yrs
[38/500] ANGELON

C:\Users\jayan\AppData\Local\Temp\ipykernel_18792\2466182516.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  both = pd.concat(frames, ignore_index=True)
C:\Users\jayan\AppData\Local\Temp\ipykernel_18792\3731490670.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(all_data).to_csv(FIN_BACKUPS_DIR / f"fundamentals_backup_{i}.csv", index=False)


  7 yrs
[51/500] ATUL    12 yrs
[52/500] AUROPHARMA    12 yrs
[53/500] AIIL    12 yrs
[54/500] AVANTIFEED    12 yrs
[55/500] DMART    12 yrs
[56/500] AXISBANK    12 yrs
[57/500] AZAD    7 yrs
[58/500] BEML    12 yrs
[59/500] BLS    12 yrs
[60/500] BSE    12 yrs
[61/500] BAGMANE    → FAILED: screener_failed
[62/500] BAJAJ-AUTO    12 yrs
[63/500] BAJFINANCE    12 yrs
[64/500] BAJAJFINSV    12 yrs
[65/500] BAJAJHLDNG    12 yrs
[66/500] BAJAJHFL    7 yrs
[67/500] BALKRISIND    12 yrs
[68/500] BALRAMCHIN    12 yrs
[69/500] BANDHANBNK    12 yrs
[70/500] BANKBARODA    12 yrs
[71/500] BANKINDIA    12 yrs
[72/500] MAHABANK    12 yrs
[73/500] BATAINDIA    12 yrs
[74/500] BELRISE    8 yrs
[75/500] BERGEPAINT    12 yrs
[76/500] BHARATCOAL    6 yrs
[77/500] BDL    12 yrs
[78/500] BEL    12 yrs
[79/500] BHARATFORG    12 yrs
[80/500] BHEL    12 yrs
[81/500] BPCL    12 yrs
[82/500] BHARTIARTL    12 yrs
[83/500] BHARTIHEXA    8 yrs
[84/500] GROWW    5 yrs
[85/500] BIOCON    12 yrs
[86/500] BSOFT    12 

C:\Users\jayan\AppData\Local\Temp\ipykernel_18792\3731490670.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(all_data).to_csv(FIN_BACKUPS_DIR / f"fundamentals_backup_{i}.csv", index=False)


  7 yrs
[102/500] CAPLIPOINT    12 yrs
[103/500] CGCL    12 yrs
[104/500] CARBORUNIV    12 yrs
[105/500] CARTRADE    12 yrs
[106/500] CASTROLIND    12 yrs
[107/500] CEATLTD    12 yrs
[108/500] CEMPRO    12 yrs
[109/500] CENTRALBK    12 yrs
[110/500] CDSL    12 yrs
[111/500] CMPDI    6 yrs
[112/500] CHAMBLFERT    12 yrs
[113/500] CHENNPETRO    12 yrs
[114/500] CHOICEIN    12 yrs
[115/500] CHOLAHLDNG    12 yrs
[116/500] CHOLAFIN    12 yrs
[117/500] CIPLA    12 yrs
[118/500] CUB    12 yrs
[119/500] CLEANMAX    7 yrs
[120/500] CLEAN    12 yrs
[121/500] COALINDIA    12 yrs
[122/500] COCHINSHIP    12 yrs
[123/500] COFORGE    12 yrs
[124/500] COHANCE    8 yrs
[125/500] COLPAL    17 yrs
[126/500] CAMS    12 yrs
[127/500] CONCORDBIO    11 yrs
[128/500] CONCOR    12 yrs
[129/500] COROMANDEL    12 yrs
[130/500] CRAFTSMAN    12 yrs
[131/500] CREDITACC    12 yrs
[132/500] CROMPTON    12 yrs
[133/500] CUMMINSIND    12 yrs
[134/500] CUPID    12 yrs
[135/500] CYIENT    12 yrs
[136/500] DLF    12 yrs
[

C:\Users\jayan\AppData\Local\Temp\ipykernel_18792\3731490670.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(all_data).to_csv(FIN_BACKUPS_DIR / f"fundamentals_backup_{i}.csv", index=False)


... (14 outputs left out)


[451/500] TENNIND    6 yrs
[452/500] THANGAMAYL    12 yrs
[453/500] NIACL    12 yrs
[454/500] RAMCOCEM    12 yrs
[455/500] THERMAX    12 yrs
[456/500] TIMKEN    12 yrs
[457/500] TITAGARH    12 yrs
[458/500] TITAN    12 yrs
[459/500] TORNTPHARM    12 yrs
[460/500] TORNTPOWER    12 yrs
[461/500] TARIL    12 yrs
[462/500] TRENT    12 yrs
[463/500] TRIDENT    12 yrs
[464/500] TRITURBINE    12 yrs
[465/500] TIINDIA    12 yrs
[466/500] UCOBANK    12 yrs
[467/500] UNOMINDA    12 yrs
[468/500] UPL    12 yrs
[469/500] UTIAMC    14 yrs
[470/500] ULTRACEMCO    12 yrs
[471/500] UNIONBANK    12 yrs
[472/500] UBL    12 yrs
[473/500] UNITDSPR    12 yrs
[474/500] URBANCO    7 yrs
[475/500] USHAMART    12 yrs
[476/500] VTL    12 yrs
[477/500] VBL    12 yrs
[478/500] VAML    → FAILED: screener_failed
[479/500] VISL    → FAILED: screener_failed
[480/500] VEDL    12 yrs
[481/500] VOGL    → FAILED: screener_failed
[482/500] VEDPOWER    → FAILED: screener_failed
[483/500] VIJAYA    9 yrs
[484/500] VMM    7

C:\Users\jayan\AppData\Local\Temp\ipykernel_18792\3731490670.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(all_data).to_csv(FIN_BACKUPS_DIR / f"fundamentals_backup_{i}.csv", index=False)


         symbol     fy_end  book_value  total_assets  operating_profitability
957        BBOX 2019-03-31        19.0        2432.0                 0.105263
2839     INDIGO 2021-03-31       111.0       43051.0               -19.423423
3356       LICI 2017-03-31       606.0     2571946.0                13.683168
3357       LICI 2018-03-31       650.0     2844989.0                14.435385
3358       LICI 2019-03-31       810.0     3427249.0                 7.308642
3359       LICI 2020-03-31      1002.0     3499834.0                -6.933134
3360       LICI 2021-03-31      6884.0     3829524.0                -0.036607
3361       LICI 2022-03-31     11337.0     4254058.0                 0.293376
3603      MSUMI 2021-03-31         0.1        1789.0              5460.000000
4249     PFOCUS 2023-03-31        25.0        6743.0                22.160000
4648  SIGNATURE 2023-03-31        47.0        6022.0                -1.425532
4905   TATACOMM 2021-03-31       115.0       21306.0            

C:\Users\jayan\AppData\Local\Temp\ipykernel_18792\3731490670.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat(all_data, ignore_index=True)


**Data limitations (fundamentals)**
- Screener gives 12 annual columns: FY2015 onward. B/M and OP start FY2015; CMA (needs 2 years) starts FY2016.
  → First full 5-factor year is Jul-2016; about 10 years of factor returns, which is short for significant premia.
- Indian GAAP → Ind AS change around FY2016–FY2018 changes book equity. Revaluation reserves cannot be removed with Screener data.
- Units: every accounting value is ₹ crore, as published by Screener. No conversion anywhere.
- Financial firms (NSE industry "Financial Services") are excluded from the SMB/HML/RMW/CMA sorts; MKT includes them.
- Basis: consolidated where available, standalone otherwise (column `basis`).

In [28]:
# =====================================================================
# STEP 6b: Availability rule (replaces get_date + monthly ffill + shift(3))
# SEBI LODR Reg 33: audited annual results within 60 days of FY end
# -> 3-month lag is safe: FY Mar-2017 is usable from 30-Jun-2017.
# =====================================================================

final_df["available_date"] = final_df["fy_end"] + pd.offsets.MonthEnd(LAG_MONTHS)

def fundamentals_as_of(fund, formation_date):
    """One row per stock: the latest fiscal year that was PUBLIC on formation_date.
    Only fiscal years ending in the 12 months up to (formation_date − lag) count.
    This is the Indian version of FF's 'fiscal year ending in calendar year t−1'."""
    
    F = pd.Timestamp(formation_date)
    oldest_ok = F - pd.offsets.MonthEnd(LAG_MONTHS + 12)
    
    usable = fund[(fund["available_date"] <= F) & (fund["fy_end"] > oldest_ok)]
    
    return usable.sort_values("fy_end").groupby("symbol").tail(1)

# Test: a June-2017 formation must pick FY Mar-2017, never Mar-2016
print(fundamentals_as_of(final_df, "2017-06-30")["fy_end"].value_counts().head())

fy_end
2017-03-31    407
2016-12-31     13
2016-06-30      1
2016-09-30      1
Name: count, dtype: int64


In [29]:
# import sys
# !{sys.executable} -m pip install plotly

In [30]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'browser'   # safest across environments

# Step 1: Count + sort years
year_counts = (
    final_df['year']
    .astype(int)
    .value_counts()
    .sort_index()
    .reset_index()
)

year_counts.columns = ['year', 'count']

# Step 2: Plot
fig = px.bar(
    year_counts,
    x='year',
    y='count',
    title='Number of Data Points per Year',
    labels={'year': 'Year', 'count': 'Data Availability: Number of stocks out of 500'}
)

# Optional styling
fig.update_layout(
    xaxis=dict(dtick=1),   # show every year
    template='plotly_white'
)

fig.show()

In [31]:
# All accounting values stay in ₹ crore, exactly as on Screener. Do NOT convert.
# (Market cap will be converted to crore in Step 7: close × shares / 1e7)

In [32]:
final_df[['operating_profit', 
        'interest_expense', 
        'depreciation', 
        'net_income', 
        'total_assets', 
        'operating_profitability',
        'book_value']].describe().transpose()


,count,mean,std,min,25%,50%,75%,max
operating_profit,4905.0,3148.365953,9520.640123,-19850.000000,267.00000,734.000000,1979.000000,179065.0
interest_expense,5606.0,2526.988049,12289.983243,-14.000000,16.00000,84.000000,509.000000,315005.0
depreciation,5606.0,815.990189,3080.423105,-11.000000,47.00000,138.000000,416.750000,57688.0
net_income,5606.0,1869.479665,6383.866426,-73878.000000,123.00000,444.000000,1320.750000,95754.0
total_assets,5606.0,82181.662861,371036.562018,0.000000,2890.25000,8413.000000,31795.750000,8321176.0
operating_profitability,4811.0,1.377656,78.718651,-19.423423,0.12899,0.217486,0.313281,5460.0
book_value,5606.0,14233.328855,42545.687773,-104167.000000,1380.00000,3519.500000,10535.250000,904030.0


In [33]:
final_df.shape

(5606, 19)

In [34]:
final_df.head()

,symbol,fy_end,year,quarter,sales,operating_profit,other_income,interest_expense,depreciation,net_income,total_assets,book_value,basis,operating_profitability,is_financial,months_since_prev_fy,prev_basis,check_me,available_date
0,360ONE,2015-03-31,2015,FY,370.0,160.0,0.0,12.0,1.0,99.0,502.0,172.0,standalone,0.860465,True,NaN,NaN,False,2015-06-30
1,360ONE,2016-03-31,2016,FY,405.0,163.0,0.0,17.0,3.0,103.0,1350.0,1140.0,standalone,0.128070,True,12.0,standalone,False,2016-06-30
2,360ONE,2017-03-31,2017,FY,444.0,159.0,0.0,18.0,8.0,86.0,1961.0,1274.0,standalone,0.110675,True,12.0,standalone,False,2017-06-30
3,360ONE,2018-03-31,2018,FY,1705.0,1053.0,10.0,559.0,14.0,380.0,9567.0,1863.0,consolidated,0.265164,True,12.0,standalone,False,2018-06-30
4,360ONE,2019-03-31,2019,FY,1578.0,990.0,1.0,431.0,22.0,375.0,9763.0,2911.0,consolidated,0.192030,True,12.0,consolidated,False,2019-06-30


In [35]:
final_df.to_csv(PROCESSED_DIR / "final_fin_data.csv", index=False)

In [36]:
price_data.head()

,date,close,traded_value,trade_days,tt_days,ret,symbol,months_traded,tv_median_12m,trade_days_12m
0,2019-10-31,1327.45,6.024053e+08,18,0,0.037273,360ONE,1,NaN,NaN
1,2019-11-30,1311.40,2.685646e+08,20,0,-0.012091,360ONE,2,NaN,NaN
2,2019-12-31,1133.85,2.139008e+08,21,0,-0.135390,360ONE,3,NaN,NaN
3,2020-01-31,1338.95,4.197572e+08,23,0,0.180888,360ONE,4,NaN,NaN
4,2020-02-29,1463.95,1.177759e+09,20,0,0.093357,360ONE,5,NaN,NaN


In [37]:
price_data.columns

Index(['date', 'close', 'traded_value', 'trade_days', 'tt_days', 'ret',
       'symbol', 'months_traded', 'tv_median_12m', 'trade_days_12m'],
      dtype='object')

In [38]:
price_data.describe().transpose()

,count,mean,min,25%,50%,75%,max,std
date,69084,2019-09-13 00:40:28.764982016,2011-01-31 00:00:00,2015-11-30 00:00:00,2020-01-31 00:00:00,2023-08-31 00:00:00,2026-09-30 00:00:00,NaN
close,68752.0,1405.606697,0.6,153.4,437.3,1142.6125,157515.0,5061.813747
traded_value,69084.0,21404198194.803715,0.0,1270773612.925,6637633514.225,22586068056.087502,1218423076969.800049,44783942154.402763
trade_days,69084.0,20.473872,0.0,20.0,21.0,22.0,23.0,2.146758
tt_days,69084.0,0.044554,0.0,0.0,0.0,0.0,23.0,0.930521
ret,68622.0,0.01648,-0.955285,-0.052064,0.007869,0.077034,1.652501,0.129228
months_traded,69084.0,85.187815,1.0,37.0,80.0,131.0,189.0,54.373198
tv_median_12m,63660.0,18958729787.693798,0.0,1238822029.11875,6377513825.875,20808678368.049999,670539464955.300049,37674402152.362709
trade_days_12m,63660.0,245.918882,0.0,246.0,248.0,249.0,252.0,13.236196


In [50]:
# =====================================================================
# STEP 7: Factor inputs (one row per stock per formation year)
# =====================================================================
import yfinance as yf

FORMATION_MONTH = 6       # portfolios formed at end of June, held July -> next June

In [51]:
# =====================================================================
# 7.1 Shares outstanding (yfinance), saved to disk so we download only once
# =====================================================================
SHARES_DIR = RAW_DIR / "shares_yf"
SHARES_DIR.mkdir(parents=True, exist_ok=True)

def get_shares(symbol):
    """Daily share counts for one stock, or None."""
    path = SHARES_DIR / f"{symbol}.csv"

    # Already downloaded before -> read from disk
    if path.exists():
        return pd.read_csv(path, parse_dates=["date"])

    try:
        s = yf.Ticker(f"{symbol}.NS").get_shares_full(start="2011-01-01")
    except Exception as e:
        print(symbol, "yfinance error:", e)
        return None

    if s is None or len(s) == 0:
        return None

    # Remove the time zone but keep the local date
    dates = pd.to_datetime(s.index)
    if dates.tz is not None:
        dates = dates.tz_localize(None)

    df = pd.DataFrame({"date": dates, "shares": s.values})
    df.to_csv(path, index=False)
    time.sleep(0.5)
    return df


In [52]:
# One value per month: the median share count reported in that month
monthly_shares = []

for sym in price_data["symbol"].unique():
    daily = get_shares(sym)
    if daily is None:
        continue
    daily = daily.set_index("date")
    one = daily["shares"].resample(MONTH_END).median().reset_index()
    one = one.rename(columns={"shares": "shares_yf"})
    one["symbol"] = sym
    monthly_shares.append(one)

shares_yf = pd.concat(monthly_shares, ignore_index=True)
shares_yf = shares_yf.dropna(subset=["shares_yf"])
print("Stocks with yfinance shares:", shares_yf["symbol"].nunique())

Stocks with yfinance shares: 495


In [53]:
# =====================================================================
# 7.2 Shares for every month + market cap (₹ crore)
# =====================================================================

price_data = price_data.merge(shares_yf, on=["symbol", "date"], how="left", validate="1:1")

# Carry a yfinance value forward at most 3 months (yfinance has gaps)
price_data = price_data.sort_values(["symbol", "date"]).reset_index(drop=True)
price_data["shares_yf"] = price_data.groupby("symbol")["shares_yf"].ffill(limit=3)


In [54]:
def fill_shares(one):
    """one = monthly rows of ONE stock, sorted by date."""
    one = one.copy()

    # Running product of split/bonus factors (halves at a 1:1 bonus)
    one["split_index"] = one["ca_factor"].cumprod()

    one["shares"] = one["shares_yf"]
    one["shares_source"] = "yfinance"

    known = one[one["shares_yf"].notna()]
    if known.empty:
        one["shares_source"] = "missing"
        return one

    # Months before the first yfinance value: undo later splits/bonuses
    first = known.iloc[0]
    before = one["date"] < first["date"]
    one.loc[before, "shares"] = first["shares_yf"] * first["split_index"] / one.loc[before, "split_index"]
    one.loc[before, "shares_source"] = "backfilled"

    one.loc[one["shares"].isna(), "shares_source"] = "missing"
    return one


In [55]:
filled = []
for sym, one in price_data.groupby("symbol"):
    filled.append(fill_shares(one))
price_data = pd.concat(filled, ignore_index=True)

# Market cap in ₹ crore: UNADJUSTED close × total shares (FF uses total shares, not free float)
price_data["me"] = price_data["close"] * price_data["shares"] / 1e7

# Last month's market cap = the weight for this month's value-weighted return
price_data["me_lag"] = price_data.groupby("symbol")["me"].shift(1)

print(price_data["shares_source"].value_counts())

shares_source
yfinance      41399
backfilled    24983
missing        2702
Name: count, dtype: int64


In [56]:
# =====================================================================
# 7.3 Per stock, per fiscal year: asset growth, ME at FY end, B/M
# =====================================================================

fund = final_df.sort_values(["symbol", "fy_end"]).reset_index(drop=True)

# --- CMA input: total-asset growth from FY t-1 to FY t ---
fund["prev_total_assets"] = fund.groupby("symbol")["total_assets"].shift(1)
fund["asset_growth"] = fund["total_assets"] / fund["prev_total_assets"] - 1


In [57]:
# Only valid if the two years are exactly 12 months apart and on the same basis
not_12_months = fund["months_since_prev_fy"] != 12
basis_changed = fund["basis"] != fund["prev_basis"]
bad_assets = fund["prev_total_assets"] <= 0
fund.loc[not_12_months | basis_changed | bad_assets, "asset_growth"] = np.nan


In [58]:
# --- Market cap at the firm's own fiscal year-end (India's version of FF's December ME) ---
me_fy = price_data[["symbol", "date", "me"]]
me_fy = me_fy.rename(columns={"date": "fy_end", "me": "me_fy_end"})
fund = fund.merge(me_fy, on=["symbol", "fy_end"], how="left", validate="1:1")


In [59]:
# --- HML input: B/M = book equity / ME, both at the same fiscal year-end, both ₹ crore ---
fund["bm"] = fund["book_value"] / fund["me_fy_end"]
fund.loc[fund["book_value"] <= 0, "bm"] = np.nan      # FF: negative book equity -> excluded
fund.loc[fund["me_fy_end"] <= 0, "bm"] = np.nan

In [60]:
# =====================================================================
# 7.4 Characteristics at each June formation date
# =====================================================================

first_year = fund["fy_end"].min().year
snapshots = []

for year in range(first_year, END_DATE.year + 1):
    F = pd.Timestamp(year=year, month=FORMATION_MONTH, day=1) + pd.offsets.MonthEnd(0)
    if F > pd.Timestamp(END_DATE):
        break
    snap = fundamentals_as_of(fund, F).copy()     # latest fiscal year that was public on F
    snap["formation_date"] = F
    snapshots.append(snap)


In [61]:
chars = pd.concat(snapshots, ignore_index=True)

# Add values measured AT the formation date: size (June ME) and the eligibility helpers
at_F = price_data[["symbol", "date", "me", "shares_source",
                   "months_traded", "tv_median_12m", "trade_days_12m"]]
at_F = at_F.rename(columns={"date": "formation_date", "me": "me_formation"})
chars = chars.merge(at_F, on=["symbol", "formation_date"], how="left", validate="1:1")

print(chars.shape)

(5598, 29)


In [62]:
# =====================================================================
# 7.5 Monthly panel: returns, Rf, weights, and which formation each month belongs to
# =====================================================================

panel = price_data[["symbol", "date", "ret", "close", "shares", "me", "me_lag",
                    "shares_source", "trade_days", "tt_days"]].copy()

panel = panel.merge(rf_monthly.reset_index(), on="date", how="left", validate="m:1")
panel["excess_ret"] = panel["ret"] - panel["Rf_monthly"]
panel["is_financial"] = panel["symbol"].isin(FIN_SYMBOLS)


In [63]:
def formation_date_for(month_end):
    """Jul-2017 ... Jun-2018 -> 2017-06-30 (the June that formed the portfolio)."""
    if month_end.month > FORMATION_MONTH:
        year = month_end.year
    else:
        year = month_end.year - 1
    return pd.Timestamp(year=year, month=FORMATION_MONTH, day=1) + pd.offsets.MonthEnd(0)


panel["formation_date"] = panel["date"].map(formation_date_for)
print(panel.shape)

(69084, 14)


In [64]:
# =====================================================================
# 7.6 Checks
# =====================================================================

In [65]:
# 1) How many stocks have each characteristic, per formation date
summary = chars.groupby("formation_date").agg(
    stocks=("symbol", "count"),
    size=("me_formation", "count"),
    bm=("bm", "count"),
    op=("operating_profitability", "count"),
    inv=("asset_growth", "count"),
)
print(summary)


                stocks  size   bm   op  inv
formation_date                             
2002-06-30           1     0    0    0    0
2006-06-30           1     0    0    1    0
2007-06-30           2     0    0    2    1
2008-06-30           2     0    0    2    1
2009-06-30           4     0    0    3    2
2010-06-30           5     0    0    4    4
2011-06-30           7     3    1    6    4
2012-06-30           6     3    3    6    5
2013-06-30           6     3    3    6    5
2014-06-30           7     1    1    6    3
2015-06-30         399   288  281  337    6
2016-06-30         416   293  289  355  383
2017-06-30         422   307  298  360  395
2018-06-30         438   305  301  370  398
2019-06-30         452   363  352  386  414
2020-06-30         476   373  365  405  436
2021-06-30         488   391  378  418  460
2022-06-30         491   415  398  421  479
2023-06-30         492   423  418  425  477
2024-06-30         493   442  429  426  487
2025-06-30         495   467  46

In [66]:
# 2) Share-count source: how much is backfilled
print(chars.groupby("formation_date")["shares_source"].value_counts().unstack())


shares_source   backfilled  missing  yfinance
formation_date                               
2011-06-30             3.0      NaN       NaN
2012-06-30             3.0      NaN       NaN
2013-06-30             3.0      NaN       NaN
2014-06-30             1.0      NaN       NaN
2015-06-30           284.0     11.0       4.0
2016-06-30           277.0     22.0      16.0
2017-06-30            55.0     23.0     253.0
2018-06-30            51.0     45.0     254.0
2019-06-30            43.0      NaN     320.0
2020-06-30            41.0      1.0     332.0
2021-06-30            38.0      NaN     354.0
2022-06-30            30.0      NaN     387.0
2023-06-30            21.0      1.0     404.0
2024-06-30            14.0      NaN     431.0
2025-06-30             5.0      1.0     462.0
2026-06-30             2.0    163.0     328.0


In [67]:
# 3) Share counts that jump >50% in a month with no split/bonus = bad yfinance data
price_data["shares_change"] = price_data.groupby("symbol")["shares"].pct_change(fill_method=None)
jumps = price_data[(price_data["shares_change"].abs() > 0.5) & (price_data["ca_factor"] == 1)]
print("Suspicious share jumps:", len(jumps))
print(jumps[["symbol", "date", "shares", "shares_change"]].head(20))


Suspicious share jumps: 162
          symbol       date        shares  shares_change
41        360ONE 2023-03-31  3.562585e+08       3.002091
218     AARTIIND 2019-10-31  1.742660e+08       1.010622
239     AARTIIND 2021-07-31  3.625140e+08       1.079995
2411  ADANIPOWER 2025-10-31  1.928729e+10       3.972014
2729       AFFLE 2021-10-31  1.332575e+08       3.998978
2999        AIIL 2026-01-31  8.499080e+08       3.894903
3660  ANANDRATHI 2025-03-31  8.304450e+07       0.999735
3675  ANANDRATHI 2026-06-30  1.660413e+08       1.000000
3932    ANGELONE 2026-02-28  4.999803e+08       4.501524
3934    ANGELONE 2026-04-30  9.108592e+08       0.821790
4319   APLAPOLLO 2020-12-31  1.246050e+08       3.998415
4329   APLAPOLLO 2021-10-31  2.498070e+08       0.996316
5381    ASHOKLEY 2025-07-31  5.873050e+09       0.999912
5686     ASTERDM 2026-07-31  8.710816e+08       0.685075
6238      AUBANK 2022-06-30  6.301860e+08       1.000568
7225  BAJAJFINSV 2022-09-30  1.593000e+09       8.999121
766

In [68]:
# 4) OP at formation dates only (each firm-year counted once), non-financials
non_fin = chars[~chars["is_financial"]]
print(non_fin["operating_profitability"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))


count    4382.000000
mean        1.486663
std        82.481311
min       -19.423423
1%         -0.443203
5%         -0.026024
50%         0.219289
95%         0.536370
99%         1.154512
max      5460.000000
Name: operating_profitability, dtype: float64


In [69]:
# 5) Tiny book equity flagged in Step 6 (MSUMI-type errors)
print(chars.loc[chars["check_me"], ["symbol", "formation_date", "book_value", "operating_profitability"]])

         symbol formation_date  book_value  operating_profitability
963        LICI     2017-06-30       606.0                13.683168
1387       LICI     2018-06-30       650.0                14.435385
1826       LICI     2019-06-30       810.0                 7.308642
1968       BBOX     2019-06-30        19.0                 0.105263
2284       LICI     2020-06-30      1002.0                -6.933134
2674      MSUMI     2021-06-30         0.1              5460.000000
2764       LICI     2021-06-30      6884.0                -0.036607
2798   TATACOMM     2021-06-30       115.0                33.400000
3059     INDIGO     2021-06-30       111.0               -19.423423
3252       LICI     2022-06-30     11337.0                 0.293376
3665     PFOCUS     2023-06-30        25.0                22.160000
3857  SIGNATURE     2023-06-30        47.0                -1.425532


In [70]:
# =====================================================================
# 7.7 Save Phase 1 outputs
# =====================================================================

chars.to_csv(PROCESSED_DIR / "characteristics_annual.csv", index=False)
panel.to_csv(PROCESSED_DIR / "monthly_panel.csv", index=False)
print("Saved: characteristics_annual.csv  (one row per stock per June formation)")
print("Saved: monthly_panel.csv           (one row per stock per month)")

Saved: characteristics_annual.csv  (one row per stock per June formation)
Saved: monthly_panel.csv           (one row per stock per month)


## Phase 1 Summary: Data Ingestion & Cleaning

| Output | Contents |
|---|---|
| `monthly_panel.csv` | Stock × month: total return (dividends, split-adjusted), Rf, excess return, ME (₹ cr), ME of previous month (weights), formation_date |
| `characteristics_annual.csv` | Stock × June formation: ME in June (size), B/M, OP, asset growth, eligibility helpers, flags |
| `rf_monthly.csv` | 91-day T-bill yield / 12, known at start of month |

**Definitions (FF5)**
- Size = ME at end of June T. ME = unadjusted close × total shares (not free float), ₹ crore.
- B/M = book equity (equity capital + reserves) / ME, both at the firm's fiscal year-end.
- OP = (operating profit − interest) / book equity, same fiscal year.
- Investment = total assets FY t / total assets FY t−1 − 1 (only if 12 months apart, same basis).
- One fiscal year per stock per formation: latest FY public by June T (3-month lag, SEBI LODR Reg 33).

**Phase 2 rules**
- 2×3 sorts: size at the median, B/M, OP, investment at 30th/70th percentiles. SMB = average of the three size spreads.
- Rebalance each June; hold July–June; value-weight with previous month's ME.
- MKT uses all stocks. The four characteristic sorts exclude financials. BE > 0 required for the HML and RMW sorts.
- Eligibility at formation: months_traded, tv_median_12m, trade_days_12m (thresholds set in Phase 2).

**Limitations**
- Fundamentals start FY2015 (Screener), so first full 5-factor year is Jul-2016.
- Universe = today's Nifty 500 (survivorship) until the bhavcopy universe replaces it.
- Shares before 2017 are backfilled from the first yfinance value using splits/bonuses only.
- Rf from FRED interbank rate until the RBI T-bill file is added (if applicable).